# Occupancy Prediction - RandomForest

Trains the classical occupancy prediction model from PIR and reservation data.

Artifacts are saved back to Google Drive under `AIModelsAndAlgorithms/OccupancyPrediction`.


In [1]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/abdullahtapanci/VisualizationApp.git'
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


Mounted at /content/drive
linked /content/VisualizationApp/Data/PIRSensorData.csv -> /content/drive/MyDrive/VisualizationApp/Data/PIRSensorData.csv
linked /content/VisualizationApp/Data/hotelReservationData.csv -> /content/drive/MyDrive/VisualizationApp/Data/hotelReservationData.csv
linked /content/VisualizationApp/Data/lightningData.csv -> /content/drive/MyDrive/VisualizationApp/Data/lightningData.csv
linked /content/VisualizationApp/Data/temperatureData.csv -> /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv
linked /content/VisualizationApp/Data/WheatherDataAntalya.csv -> /content/drive/MyDrive/VisualizationApp/Data/WheatherDataAntalya.csv
PROJECT_DIR = /content/VisualizationApp
DATA_DIR = /content/VisualizationApp/Data
DRIVE_OUTPUT_ROOT = /content/drive/MyDrive/VisualizationApp


In [ ]:
# Optional: if your new dataset files are in another Drive folder, set NEW_DATA_DIR and run this cell.
# The files will be linked/copied through DRIVE_DATA_DIR and then used by the cloned repo.
# Example: NEW_DATA_DIR = Path('/content/drive/MyDrive/NewVisualizationAppData')
NEW_DATA_DIR = None

if NEW_DATA_DIR is not None:
    for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
        src = Path(NEW_DATA_DIR) / filename
        dst = DRIVE_DATA_DIR / filename
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            !cp "{src}" "{dst}"
            print('copied', src, '->', dst)
        else:
            print('missing in NEW_DATA_DIR:', src)


In [3]:
!pip install -q pandas numpy scikit-learn joblib matplotlib


In [4]:
assert (DATA_DIR / 'PIRSensorData.csv').exists(), 'Missing PIRSensorData.csv'
assert (DATA_DIR / 'hotelReservationData.csv').exists(), 'Missing hotelReservationData.csv'
OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'AIModelsAndAlgorithms/OccupancyPrediction'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


OUTPUT_DIR = /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction


## Smoke Test
Run this first to verify paths and dependencies without training on the full dataset.


In [5]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/OccupancyPrediction/occupancy_prediction.py --max-rows 100000 --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
rows: 99,988
train rows: 79,990
test rows: 19,998
split time: 2022-10-10 17:48:00

class distribution in target:
target_room_state_1h
Occupied    0.542
Vacant      0.457
Cleaning    0.002

classification report:
              precision    recall  f1-score   support

    Cleaning      0.083     0.821     0.150        39
    Occupied      0.990     0.990     0.990      8603
      Vacant      0.992     0.962     0.977     11356

    accuracy                          0.974     19998
   macro avg      0.688     0.924     0.706     19998
weighted avg      0.990     0.974     0.981     19998

confusion matrix (rows=true, columns=predicted):
          Cleaning  Occupied  Vacant
true                                
Cleaning        32         7       0
Occupied         0      8519      84
Vacant         355        77   10924

saved /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction/occupancy_classification_report.txt
saved /content/drive/M

## Full Training
Run this cell when you are ready. It may take a long time on the full dataset.


In [6]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/OccupancyPrediction/occupancy_prediction.py  --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
^C


## Check Saved Artifacts


In [7]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)


Saved files:
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction/occupancy_classification_report.txt
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction/occupancy_confusion_matrix.csv
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction/occupancy_model.joblib
